# WSW Millisecond Checker

Iterate through folders, checking the millisecond offsets of timestamps.

Copyright 2026 Michael George (AKA Logiqx).

This file is part of [GPS Wizard](https://github.com/Logiqx/gps-wizard) and is distributed under the terms of the GNU General Public License.

GPS Wizard is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

GPS Wizard is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with GPS Wizard. If not, see <https://www.gnu.org/licenses/>.

## Notes

This is very crude and simply looks for Doppler speeds in excess of 30 m/s (i.e. 108 km/h or 58.32 knots).

Such speeds are currently unrealistic for a windsurfer.

In [1]:
import os
import sys

import numpy as np

from shapely.geometry import Polygon, Point

import traceback

corePath = os.path.join('..', 'core')
if corePath not in sys.path:
    sys.path.extend([corePath])

from file_reader import getFileReader

## Main Function

In [2]:
def findSpikes():
    '''Iterate through session archive testing each GPS file'''

    rootDir = os.path.join(projdir, '..', 'wsw-data', 'events')

    errors = {}
    
    totFiles = 0
    totPoints = 0
    totDist = 0

    for root, subDirs, files in sorted(os.walk(rootDir)):
        for file in sorted(files):
            ext = os.path.splitext(file)[1].lower()

            # Ignore SBP files because they typically contain multiple sessions
            if ext in ['.oao']:
                filePath = os.path.join(root, file)
                reader = getFileReader(filePath)
                try:
                    reader.load()
                    
                    # Process all tracks within the file
                    for track in reader.tracks:

                        # Select points where number of sats is not zero
                        points = (np.argwhere(track.data['sat'] > 0))
                        ms = ((track.data['ts'][points] % 1 * 1000).round() % 1000).astype(int)

                        # The Motion OAO files are 5 Hz and Locosys SBN files are 1 Hz
                        if ext == '.oao':
                            uniqueValues, counts = np.unique(ms, return_counts=True)
                        
                            if len(uniqueValues) > 5:
                                print(os.path.basename(root), file, uniqueValues, counts)

                            totPoints += len(track.data['sog'])
                            totDist += track.data['sog'].sum() / 5

                        else:
                            # Only interested in SBN files from the GT-31, not GT-11
                            if reader.device == 'GT-31':
                                # Ignore the first 10 seconds
                                uniqueValues, counts = np.unique(ms[10:], return_counts=True)
                            
                                if len(uniqueValues) > 1:
                                    print(os.path.basename(root), file, uniqueValues, counts)

                            totPoints += len(track.data['sog'])
                            totDist += track.data['sog'].sum()

                    totFiles += 1

                except Exception:
                    errors[filePath.replace(projdir + '/', '')] = traceback.format_exc()

    print(os.linesep * 2 + 'Summary: files = {:,}, points = {:,}, distance = {:,.02f} km'.format(
        totFiles, totPoints, totDist / 1000))

    if len(errors) > 0:
        print(os.linesep * 2 + 'Errors:')
        for error in errors:
            print(error)
            print(errors[error])

In [3]:
if __name__ == '__main__':
    projdir = os.path.realpath(os.path.join(sys.path[0], "..", ".."))

    findSpikes()
    
    print(os.linesep + 'All done!')

20241018 STA1033SCO_1033_20241018_142931.oao [  0   4   7 197 198 200 202 206 389 395 397 398 400 403 406 596 597 600
 602 605 794 800 802 806 807 982 993 994] [1975    1    1    1    2 1973    1    1    1    2    1    2 1981    2
    1    1    1 1986    2    1    2 1981    1    1    1    1    1    1]


Summary: files = 1,037, points = 27,788,140, distance = 36,906.36 km

All done!
